# Notebook 03 — Feature Engineering

## Overview

We take the cleaned, raw match data and engineer powerful, predictive features for our machine learning models. 

Raw combat and economy stats scale linearly with game duration. To make apples-to-apples comparisons between players in 20-minute stomps and 50-minute slugfests, we isolate **efficiency** and **per-minute** metrics.

### Engineered Features
1. **`kda_ratio`**: `(kills + assists) / max(deaths, 1)`
2. **`gold_per_min`**: `goldearned / (duration / 60)`
3. **`damage_efficiency`**: `totdmgtochamp / max(goldspent, 1)`
4. **`kill_participation`**: `(player_kills + player_assists) / team_total_kills`
5. **`vision_control`**: `visionscore / (duration / 60)`
6. **`death_rate`**: `deaths / (duration / 60)`
7. **`cs_per_min`**: `(totminionskilled + neutralminionskilled) / (duration / 60)`

We also **One-Hot Encode** categorical variables (`role`, `position`, `champion_classes`) and compute team-level aggregations to derive relative metrics.

### Output
A clean, numeric-heavy dataset: `data/processed/features_engineered.csv` ready for `xgboost` and `scikit-learn`.

## 0. Setup & Data Load

In [1]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

CLEAN_CSV  = '../data/processed/clean_matches.csv'
OUTPUT_CSV = '../data/processed/features_engineered.csv'

df = pd.read_csv(CLEAN_CSV)
print(f"Loaded {len(df):,} rows and {df.shape[1]} columns from {CLEAN_CSV}")
display(df.head(2))

Loaded 1,786,568 rows and 31 columns from ../data/processed/clean_matches.csv


,id,matchid,player,champion_name,champion_classes,champion_role,champion_difficulty,role,position,win,...,wardsplaced,wardskilled,duration,team_id,firsttower,firstbaron,firstdragon,towerkills,dragonkills,baronkills
0,9,10,1,Warwick,Diver,"Jungle,Top",Novice,NONE,JUNGLE,0.0,...,10.0,0.0,1909,100,1,0,0,5,0,0
1,10,10,2,Nami,Enchanter,Support,Intermediate,DUO_SUPPORT,BOT,0.0,...,17.0,3.0,1909,100,1,0,0,5,0,0


## 1. Team-Level Aggregations

To calculate a player's **Kill Participation** (KP%), we first need to count how many total kills their team secured in that match.

In [2]:
# Group by match ID and team ID to sum total team kills
team_kills = df.groupby(['matchid', 'team_id'])['kills'].sum().reset_index()
team_kills.rename(columns={'kills': 'team_total_kills'}, inplace=True)

# Merge back to main dataframe
df = pd.merge(df, team_kills, on=['matchid', 'team_id'], how='left')

# Handle edge case: teams with 0 total kills to prevent division by zero
df['team_total_kills'] = df['team_total_kills'].clip(lower=1)

print("Sample of team kills calculation: ")
display(df[['matchid', 'team_id', 'player', 'kills', 'team_total_kills']].head(5))

Sample of team kills calculation: 


,matchid,team_id,player,kills,team_total_kills
0,10,100,1,6.0,20.0
1,10,100,2,0.0,20.0
2,10,100,3,7.0,20.0
3,10,100,4,5.0,20.0
4,10,100,5,2.0,20.0


## 2. Core Feature Engineering

Compute the 7 target metrics. We use `np.where` or `.clip(lower=1)` to avoid divide-by-zero errors.

In [3]:
dur_min = df['duration'] / 60  # Duration in minutes

# 1. KDA Ratio: (K+A)/D
df['kda_ratio'] = (df['kills'] + df['assists']) / df['deaths'].clip(lower=1)

# 2. Gold Per Minute
df['gold_per_min'] = df['goldearned'] / dur_min

# 3. Damage Efficiency (Damage per gold spent)
df['damage_efficiency'] = df['totdmgtochamp'] / df['goldspent'].clip(lower=1)

# 4. Kill Participation (Percentage of team kills involved in)
df['kill_participation'] = (df['kills'] + df['assists']) / df['team_total_kills']
# Cap at 1.0 (sometimes data quirks exist)
df['kill_participation'] = df['kill_participation'].clip(upper=1.0)

# 5. Vision Control per Minute
df['vision_control'] = df['visionscore'] / dur_min

# 6. Death Rate (Deaths per minute)
df['death_rate'] = df['deaths'] / dur_min

# 7. Creep Score per minute
df['cs_per_min'] = (df['totminionskilled'] + df['neutralminionskilled']) / dur_min

engineered_cols = [
    'kda_ratio', 'gold_per_min', 'damage_efficiency', 
    'kill_participation', 'vision_control', 'death_rate', 'cs_per_min'
]

# Drop the 3 edge-case rows with NaN
df.dropna(subset=engineered_cols, inplace=True)

print("Engineered Features Summary:")
display(df[engineered_cols].describe().round(3))

print("\nSample profiles:")
display(df[['win'] + engineered_cols].head(5).round(3))

Engineered Features Summary:


,kda_ratio,gold_per_min,damage_efficiency,kill_participation,vision_control,death_rate,cs_per_min
count,1786565.000,1786565.000,1786565.000,1786565.000,1786565.000,1786565.000,1786565.000
mean,3.539,371.899,1.613,0.482,0.445,0.189,4.612
std,3.686,72.546,0.967,0.155,0.512,0.093,2.292
min,0.000,126.566,0.000,0.000,0.000,0.000,0.000
25%,1.400,320.633,1.167,0.381,0.000,0.123,3.063
50%,2.400,367.794,1.563,0.486,0.361,0.185,5.005
75%,4.200,418.718,1.996,0.587,0.689,0.249,6.331
max,46.000,978.330,884.000,1.000,3.686,1.690,14.093



Sample profiles:


,win,kda_ratio,gold_per_min,damage_efficiency,kill_participation,vision_control,death_rate,cs_per_min
0,0.0,0.700,329.921,0.886,0.35,0.440,0.314,3.489
1,0.0,6.000,298.460,1.063,0.60,0.943,0.063,0.566
2,0.0,1.500,412.865,1.225,0.60,0.817,0.251,6.537
3,0.0,0.636,345.919,1.429,0.35,0.157,0.346,5.343
4,0.0,0.500,359.529,1.739,0.20,0.471,0.251,7.512


## 3. Categorical Encoding

ML models require numeric inputs. We one-hot encode `role`, `position`, and parse out the first `champion_class` since Champions 2024.csv uses piped formats like "Mage|Support".

In [4]:
# 3.1 Map Champion Class (take the primary class only)
def extract_primary_class(c):
    if pd.isna(c) or c == 'Unknown': return 'Unknown'
    return str(c).split('|')[0]

df['primary_class'] = df['champion_classes'].apply(extract_primary_class)
print("Primary class distribution:")
print(df['primary_class'].value_counts())

# 3.2 One-Hot Encode
print("\nApplying One-Hot Encoding to role, position, and primary_class...")
cat_cols = ['role', 'position', 'primary_class']
ohe_df = pd.get_dummies(df[cat_cols], drop_first=False)

# Convert bools to int
for c in ohe_df.columns:
    ohe_df[c] = ohe_df[c].astype(int)

# Concatenate with main dataframe
df = pd.concat([df, ohe_df], axis=1)
print(f"\nAdded {ohe_df.shape[1]} dummy columns.")

Primary class distribution:
primary_class
Marksman               346827
Diver                  195997
Assassin               150716
Burst                  147744
Catcher                145907
Skirmisher             136077
Vanguard               132773
Specialist             112807
Enchanter               91862
Juggernaut              78641
Battlemage              68012
Warden                  48656
Artillery               34485
Burst  Artillery        26238
Burst  Enchanter        23344
Assassin  Diver         14681
Marksman  Catcher       13526
Marksman  Artillery      9457
Unknown                  5218
Enchanter  Warden        3597
Name: count, dtype: int64

Applying One-Hot Encoding to role, position, and primary_class...



Added 29 dummy columns.


## 4. Validation & Quality Checks

Ensure no `NaN` or `inf` values snuck into our dataset during division operations.

In [5]:
feature_set = engineered_cols + list(ohe_df.columns)

nulls = df[feature_set].isnull().sum()
print(f"Missing values: {nulls.sum()}")

infinities = np.isinf(df[engineered_cols].values).sum()
print(f"Infinity values: {infinities}")

assert nulls.sum() == 0, "NaNs found in features!"
assert infinities == 0, "Infinities found in features!"
print("\nValidation passed! All features are clean and bounded.")

Missing values: 0
Infinity values: 0

Validation passed! All features are clean and bounded.


## 5. Correlate New Features to Win

Did our feature engineering work? Let's check Pearson correlation to the target variable (`win`).

In [6]:
corr_win = df[['win'] + engineered_cols].corr()['win'].sort_values(ascending=False)

print("Pearson Correlation with 'win':")
for f, c in corr_win.items():
    if f == 'win': continue
    print(f"  {f:20} : {c:+.3f}")

Pearson Correlation with 'win':
  gold_per_min         : +0.543
  kda_ratio            : +0.485
  cs_per_min           : +0.086
  vision_control       : +0.063
  damage_efficiency    : +0.052
  kill_participation   : +0.049
  death_rate           : -0.469


## 6. Select Final Features & Save

We drop raw values (kills, duration) and keep only the refined, standardized features along with match identity handles.

In [7]:
final_cols = ['id', 'matchid', 'player', 'champion_name', 'win'] + feature_set
df_out = df[final_cols]

df_out.to_csv(OUTPUT_CSV, index=False)

size_mb = os.path.getsize(OUTPUT_CSV) / (1024**2)
print(f"\nSaved features to: {os.path.abspath(OUTPUT_CSV)}")
print(f"Dataset shape    : {df_out.shape[0]:,} rows x {df_out.shape[1]} columns")
print(f"File size        : {size_mb:.1f} MB")


Saved features to: C:\Users\asus\rift-analytics\data\processed\features_engineered.csv
Dataset shape    : 1,786,565 rows x 41 columns
File size        : 337.9 MB
